# Kubernetes — Local Setup & Deployment Testing

This notebook installs `kubectl` and `kind`, creates a local Kubernetes cluster, deploys the inference API to both namespaces, and validates the deployment — mirroring what AKS receives via CI/CD.

**Prerequisites:**
- Dev Container is running
- `bank-marketing-api:local` Docker image exists (run `04_docker_testing.ipynb` first)

---

## Why kind (not minikube)?

This project runs inside a Dev Container (Docker-outside-of-Docker). `minikube` uses SSH to bootstrap its node — that SSH connection times out in nested Docker environments. `kind` runs Kubernetes entirely via the Docker API with no SSH dependency, making it the correct tool here.

## Namespace structure (mirrors AKS)
```
kind cluster: bm-local
├── bank-marketing            ← production inference  (deployment.yaml + service.yaml)
└── bank-marketing-dev        ← staging inference     (deployment-dev.yaml + service-dev.yaml + quota-dev.yaml)
```

> **Note:** Model training runs as a `docker run` on the CI agent (or locally) — not as a Kubernetes workload. The training image is pushed to ACR for reuse by the retraining pipeline.

In [ ]:
import os

ROOT = "/workspaces/marketing-model-mlops-azure"
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()}")

## 1. Install kubectl

Downloads and installs the latest stable `kubectl` release. Skip if already installed.

In [ ]:
%%bash
if command -v kubectl &>/dev/null; then
  echo "kubectl already installed:"
  kubectl version --client
else
  echo "Installing kubectl..."
  curl -LO "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
  sudo install -o root -g root -m 0755 kubectl /usr/local/bin/kubectl
  rm kubectl
  echo "kubectl installed:"
  kubectl version --client
fi

## 2. Install kind

Downloads and installs kind v0.27.0. Skip if already installed.

In [ ]:
%%bash
if command -v kind &>/dev/null; then
  echo "kind already installed:"
  kind version
else
  echo "Installing kind v0.27.0..."
  curl -Lo /tmp/kind https://kind.sigs.k8s.io/dl/v0.27.0/kind-linux-amd64
  chmod +x /tmp/kind
  sudo mv /tmp/kind /usr/local/bin/kind
  echo "kind installed:"
  kind version
fi

## 3. Create the kind Cluster

Creates a single-node cluster named `bm-local` using `kind-config.yaml`.

> **DooD note:** `kind create cluster` will exit with an error in this Dev Container environment — this is expected. The error occurs because kind checks readiness by connecting to `localhost:<port>`, which resolves to the devcontainer, not the kind node. The cluster is fully functional; we bootstrap it manually in Section 4.
>
> `--retain` is required: without it, kind deletes the node before you can bootstrap, and you'll see `admin.conf: no such file or directory`.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

# Delete existing cluster if present
kind delete cluster --name bm-local 2>/dev/null || true

echo "Creating kind cluster (expect an error at the end — this is normal in DooD)..."
kind create cluster --config kind-config.yaml --retain || true
echo ""
echo "Cluster creation step done (proceed to Section 4 to bootstrap)."

## 4. Bootstrap the Cluster (DooD Fix)

Since kind can't verify readiness via localhost in this environment, we export the kubeconfig, patch it to use the kind node's Docker-network IP, install the CNI plugin and storage class, then wait for the node to become Ready.

> The next two cells must be run in order. The second cell waits for the API server to be reachable before installing CNI — this may take 10–30 seconds after cluster creation.

In [ ]:
%%bash
# Export kubeconfig (ignore any readiness error)
kind export kubeconfig --name bm-local

# Get the kind node's internal Docker-network IP
NODE_IP=$(docker inspect bm-local-control-plane \
  --format '{{.NetworkSettings.Networks.kind.IPAddress}}')
echo "Node IP: $NODE_IP"

# Patch kubeconfig to point at the container's IP instead of localhost
kubectl config set-cluster kind-bm-local --server=https://${NODE_IP}:6443
echo "kubeconfig patched."

In [ ]:
%%bash
# Wait for the API server to be reachable (may still be starting after cluster creation)
echo "Waiting for API server to be reachable..."
for i in $(seq 1 30); do
  kubectl get nodes &>/dev/null && break
  echo "  attempt $i — API server not ready, retrying in 3s..."
  sleep 3
done

# Install the bundled CNI plugin (kind can't do this itself in DooD)
echo "Installing CNI plugin..."
docker exec bm-local-control-plane cat /kind/manifests/default-cni.yaml \
  | sed 's/{{ .PodSubnet }}/10.244.0.0\/24/' \
  | kubectl apply -f -

# Install the default storage class
echo "Installing storage class..."
docker exec bm-local-control-plane cat /kind/manifests/default-storage.yaml \
  | kubectl apply -f -

# Remove the control-plane taint so workload pods can schedule on the single node
# (kind normally does this automatically, but the DooD error prevents it)
echo ""
echo "Removing control-plane taint..."
kubectl taint nodes --all node-role.kubernetes.io/control-plane- 2>/dev/null || true

echo ""
echo "Waiting for node to become Ready (up to 120s)..."
kubectl wait --for=condition=Ready node --all --timeout=120s

echo ""
echo "=== Cluster info ==="
kubectl cluster-info --context kind-bm-local

echo ""
echo "=== Nodes ==="
kubectl get nodes

## 5. Validate K8s Manifests with kubeconform

Client-side schema validation against official Kubernetes JSON schemas — same check that runs in CI.

In [ ]:
%%bash
if ! command -v kubeconform &>/dev/null; then
  echo "Installing kubeconform..."
  curl -sLO https://github.com/yannh/kubeconform/releases/latest/download/kubeconform-linux-amd64.tar.gz
  tar xzf kubeconform-linux-amd64.tar.gz
  sudo mv kubeconform /usr/local/bin/
  rm -f kubeconform-linux-amd64.tar.gz LICENSE
  echo "kubeconform installed."
else
  echo "kubeconform already installed."
fi

echo ""
echo "=== Validating k8s/ manifests ==="
cd /workspaces/marketing-model-mlops-azure
kubeconform -summary -strict k8s/

## 6. Load Image into kind

Loads the locally-built Docker image into the kind cluster, bypassing the need for a registry.

In [ ]:
%%bash
echo "Loading bank-marketing-api:local into kind cluster..."
kind load docker-image bank-marketing-api:local --name bm-local
echo "Image loaded."

## 7. Create Namespaces

Create both namespaces mirroring the AKS production setup.

In [ ]:
%%bash
kubectl create namespace bank-marketing 2>/dev/null || echo "Namespace bank-marketing already exists."
kubectl create namespace bank-marketing-dev 2>/dev/null || echo "Namespace bank-marketing-dev already exists."

echo ""
kubectl get namespaces | grep bank-marketing

## 8. Create Secrets & Copy Model Artifact

The deployment manifests reference Kubernetes secrets that don't exist in the kind cluster:
- **`bank-marketing-api-key`** — `API_KEY` env var for authentication (both namespaces)
- **`azure-storage`** — Azure Blob credentials (production only)

We also need the trained model artifact on the kind node — the inference image loads it from a `hostPath` volume mount (not baked into the Docker image).

> On AKS these secrets are created by CI/CD from pipeline variables / Azure Key Vault. Here we use dummy values.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

echo "=== Creating secrets ==="

# API key secret — both namespaces
kubectl create secret generic bank-marketing-api-key \
  --from-literal=API_KEY=local-dev-key \
  -n bank-marketing 2>/dev/null || echo "Secret bank-marketing-api-key already exists in bank-marketing."
kubectl create secret generic bank-marketing-api-key \
  --from-literal=API_KEY=local-dev-key \
  -n bank-marketing-dev 2>/dev/null || echo "Secret bank-marketing-api-key already exists in bank-marketing-dev."

# Azure storage secret — production only (deployment.yaml references it)
kubectl create secret generic azure-storage \
  --from-literal=ACCOUNT_NAME=dummy \
  --from-literal=CONTAINER_NAME=dummy \
  -n bank-marketing 2>/dev/null || echo "Secret azure-storage already exists in bank-marketing."

echo ""
echo "=== Secrets ==="
kubectl get secrets -n bank-marketing
echo "---"
kubectl get secrets -n bank-marketing-dev

echo ""
echo "=== Copying model artifact to kind node ==="
# The inference image expects model.pkl at /app/artifacts/ via hostPath volume
docker exec bm-local-control-plane mkdir -p /tmp/bank-marketing/artifacts
cat artifacts/model.pkl | docker exec -i bm-local-control-plane \
  sh -c "cat > /tmp/bank-marketing/artifacts/model.pkl"

echo "Verifying..."
docker exec bm-local-control-plane ls -la /tmp/bank-marketing/artifacts/
echo "Done."

## 9. Deploy to Production Namespace

Applies the production Deployment and Service, then overrides the ACR image reference with the local tag.

> **Why the scale-to-0 + JSON patch pattern?** The YAML files reference ACR images (they're the source-of-truth for AKS). In kind there's no ACR, so `kubectl apply` creates pods that immediately enter `ImagePullBackOff`. Those stuck pods consume node resources, and any subsequent `kubectl set image` or `kubectl patch` triggers a *new* rollout that deadlocks (`maxUnavailable: 0` won't terminate old pods until new ones are Ready). Scaling to 0 clears the stuck pods, letting the patch apply cleanly. Scaling back up creates pods with the correct local image from the start.
>
> The JSON patch also: sets `imagePullPolicy: Never` (use locally loaded image), overrides `env` to use `STORAGE_BACKEND=local` (no Azure Blob in kind), adds a `hostPath` volume mount for the model artifact, and removes `imagePullSecrets` (the `acr-secret` doesn't exist in kind).

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

echo "Deploying to bank-marketing (production)..."

# Apply manifests, then immediately scale to 0 to prevent stuck ACR-image pods
kubectl apply -f k8s/deployment.yaml -f k8s/service.yaml -n bank-marketing
kubectl scale deployment/bank-marketing-api --replicas=0 -n bank-marketing

# JSON patch: local image, local storage, hostPath volume, remove imagePullSecrets
kubectl patch deployment bank-marketing-api -n bank-marketing --type=json -p '[
  {"op": "replace", "path": "/spec/template/spec/containers/0/image", "value": "bank-marketing-api:local"},
  {"op": "add", "path": "/spec/template/spec/containers/0/imagePullPolicy", "value": "Never"},
  {"op": "replace", "path": "/spec/template/spec/containers/0/env", "value": [
    {"name": "UVICORN_WORKERS", "value": "1"},
    {"name": "API_KEY", "valueFrom": {"secretKeyRef": {"name": "bank-marketing-api-key", "key": "API_KEY"}}},
    {"name": "STORAGE_BACKEND", "value": "local"}
  ]},
  {"op": "add", "path": "/spec/template/spec/containers/0/volumeMounts", "value": [
    {"name": "model-artifacts", "mountPath": "/app/artifacts", "readOnly": true}
  ]},
  {"op": "add", "path": "/spec/template/spec/volumes", "value": [
    {"name": "model-artifacts", "hostPath": {"path": "/tmp/bank-marketing/artifacts", "type": "DirectoryOrCreate"}}
  ]},
  {"op": "remove", "path": "/spec/template/spec/imagePullSecrets"}
]'

# Scale back up — pods start with the correct local image
kubectl scale deployment/bank-marketing-api --replicas=2 -n bank-marketing

echo ""
echo "Waiting for rollout to complete (up to 120s)..."
kubectl rollout status deployment/bank-marketing-api -n bank-marketing --timeout=120s

echo ""
echo "=== Pods ==="
kubectl get pods -n bank-marketing

echo ""
echo "=== Service ==="
kubectl get svc -n bank-marketing

## 10. Deploy to Staging Namespace

Applies the ResourceQuota, staging Deployment, and ClusterIP Service — then patches the image using the same scale-to-0 + JSON patch pattern as production. The dev manifest also references `azure-storage` secrets and `imagePullSecrets` that don't exist in kind, so the same full patch is needed.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

echo "Deploying to bank-marketing-dev (staging)..."
kubectl apply -f k8s/quota-dev.yaml -n bank-marketing-dev
kubectl apply -f k8s/deployment-dev.yaml -f k8s/service-dev.yaml -n bank-marketing-dev

# Scale to 0 first — avoids stuck pods pulling from ACR
kubectl scale deployment/bank-marketing-api --replicas=0 -n bank-marketing-dev

# JSON patch: local image, local storage, hostPath volume, remove imagePullSecrets
# The dev manifest references azure-storage secretKeyRef which doesn't exist in kind.
kubectl patch deployment bank-marketing-api -n bank-marketing-dev --type=json -p '[
  {"op": "replace", "path": "/spec/template/spec/containers/0/image", "value": "bank-marketing-api:local"},
  {"op": "add", "path": "/spec/template/spec/containers/0/imagePullPolicy", "value": "Never"},
  {"op": "replace", "path": "/spec/template/spec/containers/0/env", "value": [
    {"name": "UVICORN_WORKERS", "value": "1"},
    {"name": "API_KEY", "valueFrom": {"secretKeyRef": {"name": "bank-marketing-api-key", "key": "API_KEY"}}},
    {"name": "STORAGE_BACKEND", "value": "local"}
  ]},
  {"op": "add", "path": "/spec/template/spec/containers/0/volumeMounts", "value": [
    {"name": "model-artifacts", "mountPath": "/app/artifacts", "readOnly": true}
  ]},
  {"op": "add", "path": "/spec/template/spec/volumes", "value": [
    {"name": "model-artifacts", "hostPath": {"path": "/tmp/bank-marketing/artifacts", "type": "DirectoryOrCreate"}}
  ]},
  {"op": "remove", "path": "/spec/template/spec/imagePullSecrets"}
]'

# Scale back up — pod starts with the correct local image
kubectl scale deployment/bank-marketing-api --replicas=1 -n bank-marketing-dev

echo ""
echo "Waiting for rollout to complete (up to 120s)..."
kubectl rollout status deployment/bank-marketing-api -n bank-marketing-dev --timeout=120s

echo ""
echo "=== Pods ==="
kubectl get pods -n bank-marketing-dev

echo ""
echo "=== Service ==="
kubectl get svc -n bank-marketing-dev

## 11. Verify ResourceQuota (Staging)

Confirm the staging quota is enforced. Used values must be at or below Hard limits.

In [ ]:
%%bash
kubectl describe resourcequota bank-marketing-dev-quota -n bank-marketing-dev

## 12. Simulate CD_Dev Smoke Test

Replicates **exactly** what the `CD_Dev` CI stage does on AKS — an exec-based health check and prediction from inside the running pod. If these pass locally, the CI stage will pass on AKS.

> **Note:** The `X-API-Key` header is required because the `bank-marketing-api-key` secret sets a non-empty `API_KEY` env var, enabling authentication.

In [ ]:
%%bash
echo "=== CD_Dev smoke test simulation ==="
echo ""

# Wait for the staging pod to be ready (mirrors docs Section 9)
echo "Waiting for staging pod to be ready..."
kubectl wait --for=condition=ready pod \
  -l app=bank-marketing-api \
  -n bank-marketing-dev \
  --timeout=120s

echo ""
echo "--- Health check (from inside pod) ---"
kubectl exec -n bank-marketing-dev \
  deploy/bank-marketing-api -- \
  curl -sf http://localhost:8000/health

echo ""
echo ""
echo "--- Prediction (from inside pod) ---"
kubectl exec -n bank-marketing-dev \
  deploy/bank-marketing-api -- \
  curl -sf -X POST http://localhost:8000/predict \
    -H "Content-Type: application/json" \
    -H "X-API-Key: local-dev-key" \
    -d '{"age":35,"job":"management","marital":"married","education":"tertiary","default":"no","balance":1500.0,"housing":"yes","loan":"no","contact":"cellular","day":15,"month":"may","duration":250.0,"campaign":1,"pdays":-1,"previous":0,"poutcome":"unknown"}'

echo ""
echo ""
echo "Smoke test complete."

## 13. Access Services via Port-Forward

Port-forward runs in the foreground — run each command in a separate terminal, or use the background option below.

> To access from the notebook, use the exec-based approach in Section 12 instead.

In [ ]:
# Print the port-forward commands to copy into a terminal.
# These cannot be run in a notebook cell (they block the foreground).
print("Run these in a terminal to access each namespace:")
print()
print("# Production namespace (LoadBalancer → port-forward):")
print("kubectl port-forward svc/bank-marketing-api 8000:80 -n bank-marketing")
print("# Then: curl http://localhost:8000/health")
print()
print("# Staging namespace (ClusterIP → port-forward):")
print("kubectl port-forward svc/bank-marketing-api 8001:8000 -n bank-marketing-dev")
print("# Then: curl http://localhost:8001/health")

---

## Summary

| Step | Tool | Expected result |
|---|---|---|
| Install kubectl | `curl` + `sudo install` | `kubectl version --client` prints version |
| Install kind | `curl` + `sudo mv` | `kind version` prints version |
| Create cluster | `kind create cluster --retain` | Error expected (DooD) — proceed to bootstrap |
| Bootstrap | `kind export kubeconfig` + IP patch + CNI + taint removal | `kubectl get nodes` shows Ready |
| Validate manifests | `kubeconform -strict k8s/` | 0 errors |
| Load inference image | `kind load docker-image` | Image available in cluster |
| Create namespaces | `kubectl create namespace` | Both namespaces exist |
| Create secrets + copy model | `kubectl create secret` + `docker exec` | Secrets in both ns, model.pkl on node |
| Deploy production | `apply` → `scale 0` → `patch --type=json` → `scale 2` | 2 Pods Running |
| Deploy staging | `apply` → `scale 0` → `patch --type=json` → `scale 1` | 1 Pod Running |
| ResourceQuota | `kubectl describe resourcequota` | Used ≤ Hard |
| CD_Dev smoke test | `kubectl wait` + `kubectl exec ... curl` | Health + predict pass |

### Namespace Mapping

| Namespace | Purpose | Triggered by |
|---|---|---|
| `bank-marketing` | Production inference | Push to `main` |
| `bank-marketing-dev` | Staging inference | Push to `dev` |

When all steps pass locally, the same configuration deploys cleanly to AKS via CI/CD.